# FlyBrain Robot Bridge: the MaleCNS connectome in 10 minutes

This notebook downloads the public MaleCNS v1.0 connectome (1.1 GB, CC BY 4.0, HHMI Janelia
and collaborators), compiles it into a spiking model of all 166,700 neurons, and drives it
with synthetic camera scenes. It then runs the same scenes on **shuffled wiring**: the same
neurons, transmitters and synapse counts, with synaptic targets permuted.

Measured on a laptop, compilation peaks at about 4.3 GB of memory, which fits a standard
Colab runtime. A GPU runtime is optional; results are identical on CPU and GPU.
The full, reproducible report is `docs/VALIDATION.md` in the repository.

In [ ]:
!git clone -q https://github.com/Frankweb33/flybrain-robot-bridge.git
%cd flybrain-robot-bridge
!pip install -q -e ".[connectome]"
import sys

sys.path.insert(0, "src")  # use the checkout without restarting the runtime

Optional, on a GPU runtime (Colab ships CUDA 12 drivers):

In [ ]:
# !pip install -q "numba-cuda-mlir[cu12]>=0.5.2"

## 1. Download and compile the connectome

In [ ]:
from flybrain_robot.connectome import build

DATA = "data/malecns-v1.0"
build.download(DATA)  # checks the pinned SHA-256 of every file
manifest = build.compile_graph(DATA, verify=False)  # already verified by download
{k: manifest[k] for k in ("neurons", "neuron_to_neuron_edges", "synapses")}

## 2. Closed loop: camera scene → connectome → motor commands

In [ ]:
import numpy as np

from flybrain_robot.brain.malecns import MaleCNSBackend
from flybrain_robot.telemetry import Telemetry
from flybrain_robot.vision import TOUR_SEGMENT, scene_frame


def run(backend, scene):
    backend.reset()
    encoder = backend.make_encoder()
    rows = []
    for i in range(TOUR_SEGMENT):
        a = backend.step(encoder.encode(scene_frame(scene, i)), Telemetry(), 1 / 30)
        r = backend.named_rates()
        rows.append((r["DNp01:L"], r["DNp01:R"], r["DNa02:L"], r["DNa02:R"],
                     (a.right - a.left) / 2, a.escape))
    return np.array(rows)

real = MaleCNSBackend(DATA, seed=0)
shuffled = MaleCNSBackend(DATA, seed=1, wiring="shuffled")
print("simulating on", real.device)
scenes = ["pass_left", "approach_left"]
results = {(w, s): run(b, s) for w, b in (("MaleCNS", real), ("shuffled", shuffled)) for s in scenes}
for (wiring, scene), r in results.items():
    print(f"{wiring:9s} {scene:14s} peak Giant Fiber {r[:, :2].max():5.0f} Hz, "
          f"peak DNa02 {r[:, 2:4].max():5.0f} Hz, mean turn {r[:, 4].mean():+.2f}")

In [ ]:
import matplotlib.pyplot as plt

t = np.arange(TOUR_SEGMENT) / 30
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
for col, scene in enumerate(scenes):
    for wiring, style in (("MaleCNS", "-"), ("shuffled", "--")):
        r = results[(wiring, scene)]
        axes[0, col].plot(t, r[:, :2].max(axis=1), style, label=f"Giant Fiber, {wiring}")
        axes[1, col].plot(t, r[:, 4], style, label=f"turn (+ left), {wiring}")
    axes[0, col].set_title(scene)
    axes[0, col].set_ylabel("Hz")
    axes[1, col].set_xlabel("s")
    axes[0, col].legend(fontsize=8)
    axes[1, col].legend(fontsize=8)
plt.tight_layout()

## What this shows and what it does not

On the real wiring a passing object turns the robot through DNa02 and an approaching one
drives the Giant Fiber, the fly's escape neuron; on shuffled wiring both vanish. This is a
leaky integrate-and-fire approximation with hand-chosen sensory and motor interfaces, run on
synthetic scenes; it is not a recording of a fly or of a physical robot. See the README's
status table and `docs/VALIDATION.md` for what has and has not been validated.